# Radical-Aligned Structure — Kaggle fast run

Free, ~30 min total on a Kaggle P100 / T4 x2.

Before running:
1. In Kaggle, click **Settings → Accelerator → GPU P100** (free tier gets you 30 hours/week).
2. Settings → Internet → On.
3. Run cells top-to-bottom.

When it finishes, the right-hand File browser will have `/kaggle/working/results/` and `/kaggle/working/figures/` — download those two folders, drop them into your local `chinese_llm_composition/` directory, and you're done.

In [ ]:
# 1. Clone your repo.
!git clone https://github.com/aryan35790jp/chinese_llm.git /kaggle/working/repo
%cd /kaggle/working/repo

In [ ]:
# 2. Install the lighter, GPU-friendly dependency stack.
#    Kaggle ships torch + CUDA preinstalled, so we only top up what's missing.
!pip install -q transformers==4.44.2 tokenizers==0.19.1 \
    numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1 \
    matplotlib==3.9.0 tqdm==4.66.4 datasets==2.20.0 sentencepiece==0.2.0 \
    Pillow==10.4.0 umap-learn==0.5.6

In [ ]:
# 3. Confirm the GPU.
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('bf16:', torch.cuda.is_bf16_supported())

In [ ]:
# 4. Smoke tests — ~30 seconds.
!python scripts/new/_check_syntax.py
!python scripts/new/_smoke_test.py

In [ ]:
# 5. Download Unihan + build the radical dataset (~3 min).
!mkdir -p data/unihan
!curl -sL -o data/unihan/Unihan.zip https://www.unicode.org/Public/UCD/latest/ucd/Unihan.zip
!cd data/unihan && unzip -o -q Unihan.zip && cd ../..
!python scripts/new/dataset_builder.py
!python scripts/new/classify_liushu.py

In [ ]:
# 6. Tokenization audit (~2 min, sets up coverage table for the appendix).
import os
os.environ['RADICAL_FAST'] = '1'
!python scripts/new/tokenization_audit.py

In [ ]:
# 7. The heavy step: extract embeddings for the 5-model fast subset.
#    ~25 min on P100, ~35 min on T4 x2.
#    bf16 inference, batch size 256, ~5 layers per model.
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/extract_embeddings.py

In [ ]:
# 8. Build the pure-vision baseline (rendered PNG → frozen ResNet-18). ~3 min.
#    Kaggle ships Noto CJK; the script auto-detects.
!apt-get install -y -q fonts-noto-cjk 2>/dev/null || true
!python scripts/new/glyph_only_baseline.py

In [ ]:
# 9. Isotropy correction (~3 min).
!python scripts/new/isotropy_correction.py

In [ ]:
# 10. All the cheap analysis steps. ~15 min total.
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/full_pipeline.py --fast --skip extract,glyph_baseline,isotropy

In [ ]:
# 11. Show the auto-generated report.
with open('results/_REPORT.md', encoding='utf-8') as f:
    from IPython.display import Markdown
    md = f.read()
Markdown(md)

In [ ]:
# 12. Pack results + figures so you can download them in one click.
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip results figures data/radical_dataset.csv data/radical_summary.csv
import os
print('Output:', '/kaggle/working/results.zip')
print('Size  :', round(os.path.getsize('/kaggle/working/results.zip')/1e6, 1), 'MB')